In [ ]:
import mteb
import pandas as pd
import os
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv

load_dotenv()

/Users/lucasokamura/miniconda3/envs/probing/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"
DEVICE

In [2]:
tasks = mteb.get_tasks(languages=["por"], modalities=['text'])

for task in tasks:
    print(task)

BibleNLPBitextMining(name='BibleNLPBitextMining', languages=['eng', 'por'])
FloresBitextMining(name='FloresBitextMining', languages=['ace', 'acm', 'acq', '...'])
NTREXBitextMining(name='NTREXBitextMining', languages=['arb', 'ben', 'cat', '...'])
TatoebaBitextMining(name='Tatoeba', languages=['eng', 'por'])
WebFAQBitextMiningQAs(name='WebFAQBitextMiningQAs', languages=['cat', 'dan', 'deu', '...'])
WebFAQBitextMiningQuestions(name='WebFAQBitextMiningQuestions', languages=['cat', 'dan', 'deu', '...'])
AfriSentiClassification(name='AfriSentiClassification', languages=['por'])
AfriSentiLangClassification(name='AfriSentiLangClassification', languages=['amh', 'arq', 'ary', '...'])
LanguageClassification(name='LanguageClassification', languages=['ara', 'bul', 'cmn', '...'])
MassiveIntentClassification(name='MassiveIntentClassification', languages=['por'])
MassiveScenarioClassification(name='MassiveScenarioClassification', languages=['por'])
MultiHateClassification(name='MultiHateClassification

In [3]:
model_name = "neuralmind/bert-base-portuguese-cased"
tasks = [
    ("Assin2STS", None),
    ("SICK-BR-STS", None),
    ("STSBenchmarkMultilingualSTS", 'pt'),
    
    ('MassiveIntentClassification', 'pt'),
    ('MultiHateClassification', 'por'),
    ('BrazilianToxicTweetsClassification', None),
    ('HateSpeechPortugueseClassification', None),
    ('TweetSentimentClassification', 'portuguese'),

    ('MultiLongDocReranking', 'pt'),
    ('WikipediaRerankingMultilingual', 'pt'),
    ('XGlueWPRReranking', 'pt'),

    ('WebFAQRetrieval', 'por'),
    ('MultiLongDocRetrieval', 'pt'),
    ('WikipediaRetrievalMultilingual', 'pt')
]

In [ ]:
# model_meta = mteb.get_model(model_name, device="cuda")
model_meta = SentenceTransformer(model_name, device=DEVICE)

# select the desired tasks and evaluate
task_name_list = []
model_name_list = []
main_score_list = []

for task_info in tasks:
    print(f"""
    #############################

    Avaliando {task_info[0]} ({task_info[1]})...
    
    #############################
    """)

    task = mteb.get_task(task_info[0], languages=['por'], hf_subsets=task_info[1])

    # with encode kwargs
    result = mteb.evaluate(model_meta, task, encode_kwargs={"batch_size": 512})

    task_name = result.task_results[0].task_name
    model_name = result.model_name
    main_score = result.task_results[0].main_score

    task_name_list.append(task_name)
    model_name_list.append(model_name)
    main_score_list.append(main_score)

print("Avaliação concluída!")

No sentence-transformers model found with name neuralmind/bert-base-portuguese-cased. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57826.42it/s]
BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    #############################

    Avaliando Assin2STS (None)...

    #############################
    

    #############################

    Avaliando SICK-BR-STS (None)...

    #############################
    

    #############################

    Avaliando STSBenchmarkMultilingualSTS (pt)...

    #############################
    

    #############################

    Avaliando MassiveIntentClassification (pt)...

    #############################
    

    #############################

    Avaliando MultiHateClassification (por)...

    #############################
    

    #############################

    Avaliando BrazilianToxicTweetsClassification (None)...

    #############################
    


Couldn't subsample, continuing with the entire test set.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
or set the environment variable OPENBLAS_NUM_THREADS to 128 or lower
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
or set the environment variable OPENBLAS_NUM_THREADS to 128 or lower
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
or set the environment variable OPENBLAS_NUM_THREADS to 128 or lower
/Users/lucasokamura/miniconda3/envs/probing/lib/python3.12/site-packages/mteb/_evaluators/classification_metrics.py:61: RuntimeWarning: invalid value encountered in d

BLAS : Bad memory unallocation! :  768  0x5c3ab8000


In [ ]:
df_results = pd.DataFrame({
    'model_name': model_name_list,
    'task_name': task_name_list,
    'main_score': main_score_list
})

NameError: name 'pd' is not defined

In [ ]:
filepath = '../data/results_eval_mteb.csv'

if os.path.exists(filepath):
    df_results_cache = pd.read_csv(filepath)
    df_results = pd.concat([df_results_cache, df_results], axis=0, ignore_index=True)

df_results.to_csv(filepath, index=False)